# 🃏 Card Deblur — NAFNet Training (Fixed)

## Checklist trước khi chạy
- [ ] Đã chạy `step2_gen_dataset.py` (version fixed) trên máy local
- [ ] Đã nén thành `data_fold0.zip` và upload lên Google Drive
- [ ] Runtime → Change runtime type → **T4 GPU**

## Các thay đổi so với version cũ
| Setting | Cũ | Mới | Lý do |
|---|---|---|---|
| batch_size | 4 | 8 | T4 đủ VRAM, hội tụ nhanh hơn |
| total_iter | 50,000 | 100,000 | Dataset nhỏ cần nhiều iter hơn |
| warmup_iter | -1 | 2000 | Tránh loss spike đầu |
| gt_size | 256 | 256 | Giữ nguyên (phù hợp kernel nhỏ mới) |
| save_checkpoint_freq | 5000 | 5000 | Giữ nguyên |
| val_freq | 2000 | 2000 | Giữ nguyên |

## Ô 1: Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
assert torch.cuda.is_available(), '❌ GPU chưa bật! Runtime → Change runtime type → T4'

## Ô 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder = '/content/drive/MyDrive/deblur_cards_project'
if os.path.exists(folder):
    print('✅ Thấy folder, nội dung:', os.listdir(folder))
else:
    print('❌ Không thấy folder! Kiểm tra tên folder trên Drive.')
    print('Nội dung MyDrive:', os.listdir('/content/drive/MyDrive'))

## Ô 3: Giải nén dataset

In [ ]:
import zipfile, os, glob

# Đường dẫn tới file zip trên Drive
ZIP_PATH  = '/content/drive/MyDrive/deblur_cards_project/data_fold0.zip'
DEST_DIR  = '/content/deblur_cards'

os.makedirs(DEST_DIR, exist_ok=True)

print('Đang giải nén...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DEST_DIR)
print('Xong!')

blur  = glob.glob(f'{DEST_DIR}/data/train/blur/*.png')
sharp = glob.glob(f'{DEST_DIR}/data/train/sharp/*.png')
tblur  = glob.glob(f'{DEST_DIR}/data/test/blur/*.png')
tsharp = glob.glob(f'{DEST_DIR}/data/test/sharp/*.png')

print(f'Train blur  : {len(blur)}')
print(f'Train sharp : {len(sharp)}')
print(f'Test blur   : {len(tblur)}')
print(f'Test sharp  : {len(tsharp)}')

assert len(blur) > 0 and len(blur) == len(sharp), '❌ Dataset lỗi!'
print('✅ Dataset OK')

## Ô 4: Cài NAFNet

In [ ]:
%cd /content
!git clone https://github.com/megvii-research/NAFNet.git
%cd /content/NAFNet
!pip install -r requirements.txt -q
!python setup.py develop --no_cuda_ext

# Verify
import sys
sys.path.insert(0, '/content/NAFNet')
from basicsr.models.archs.NAFNet_arch import NAFNet
print('✅ NAFNet import OK')

# Quick model test
import torch
m = NAFNet(img_channel=3, width=32, middle_blk_num=12,
           enc_blk_nums=[2,2,4,8], dec_blk_nums=[2,2,2,2])
x = torch.randn(1, 3, 256, 256)
y = m(x)
assert y.shape == x.shape
print(f'✅ Model forward OK: {x.shape} → {y.shape}')
del m, x, y

## Ô 5: Tải pretrained NAFNet-GoPro

In [ ]:
import os
os.makedirs('/content/pretrained', exist_ok=True)

# NAFNet-GoPro-width32 pretrained (fine-tune từ đây)
!gdown 'https://drive.google.com/uc?id=1Fr2QadtDCEXg6iwWX8OzeZLbHOx2t5Bj' \
       -O /content/pretrained/NAFNet-GoPro-width32.pth

size = os.path.getsize('/content/pretrained/NAFNet-GoPro-width32.pth') / 1e6
print(f'✅ Pretrained: {size:.1f} MB')
assert size > 50, '❌ File quá nhỏ, tải bị lỗi!'

## Ô 6: Tạo file config YAML (Fixed)

In [ ]:
# ================================================================
# CONFIG FIXED:
#   batch_size: 4 → 8    (T4 16GB đủ VRAM)
#   total_iter: 50k → 100k  (train kỹ hơn)
#   warmup_iter: -1 → 2000  (ổn định early training)
#   lr: 0.0002 → 0.0003     (batch tăng → lr tăng)
# ================================================================

import os
os.makedirs('/content/NAFNet/options/train/', exist_ok=True)

CONFIG_YAML = '''
name: NAFNet-Cards-width32
model_type: ImageRestorationModel
scale: 1
num_gpu: 1
manual_seed: 42

datasets:
  train:
    name: CardTrain
    type: PairedImageDataset
    dataroot_gt:  /content/deblur_cards/data/train/sharp
    dataroot_lq:  /content/deblur_cards/data/train/blur
    geometric_augs: true
    filename_tmpl: "{}"
    io_backend:
      type: disk

    gt_size: 256
    use_flip: true
    use_rot:  true
    num_worker_per_gpu: 4
    batch_size_per_gpu: 8
    dataset_enlarge_ratio: 1
    prefetch_mode: ~

  val:
    name: CardVal
    type: PairedImageDataset
    dataroot_gt:  /content/deblur_cards/data/test/sharp
    dataroot_lq:  /content/deblur_cards/data/test/blur
    io_backend:
      type: disk

network_g:
  type: NAFNet
  img_channel: 3
  width: 32
  enc_blk_nums: [2, 2, 4, 8]
  middle_blk_num: 12
  dec_blk_nums: [2, 2, 2, 2]

path:
  pretrain_network_g: /content/pretrained/NAFNet-GoPro-width32.pth
  strict_load_g: false
  resume_state: ~

train:
  total_iter: 100000
  warmup_iter: 2000
  use_grad_clip: true

  scheduler:
    type: TrueCosineAnnealingLR
    T_max: 100000
    eta_min: !!float 1e-7

  optim_g:
    type: AdamW
    lr: !!float 3e-4
    weight_decay: !!float 1e-3
    betas: [0.9, 0.9]

  pixel_opt:
    type: PSNRLoss
    loss_weight: 1
    reduction: mean

val:
  val_freq: !!float 2000
  save_img: false

  metrics:
    psnr:
      type: calculate_psnr
      crop_border: 0
      test_y_channel: false
    ssim:
      type: calculate_ssim
      crop_border: 0
      test_y_channel: false

logger:
  print_freq: 200
  save_checkpoint_freq: !!float 5000
  use_tb_logger: false

dist_params:
  backend: nccl
  port: 29500
'''

with open('/content/NAFNet/options/train/nafnet_custom.yml', 'w') as f:
    f.write(CONFIG_YAML.strip())

print('✅ Config đã tạo:')
print('   batch_size_per_gpu : 8')
print('   total_iter         : 100,000')
print('   warmup_iter        : 2,000')
print('   lr                 : 3e-4')
print()
!cat /content/NAFNet/options/train/nafnet_custom.yml

## Ô 7: (Chỉ chạy lần đầu) Kiểm tra VRAM với batch_size=8
Nếu VRAM không đủ → giảm batch_size về 4 trong Ô 6

In [ ]:
import torch, sys
sys.path.insert(0, '/content/NAFNet')
from basicsr.models.archs.NAFNet_arch import NAFNet

device = torch.device('cuda')
model  = NAFNet(img_channel=3, width=32, middle_blk_num=12,
                enc_blk_nums=[2,2,4,8], dec_blk_nums=[2,2,2,2]).to(device)

# Test batch_size=8, patch 256x256
try:
    x = torch.randn(8, 3, 256, 256).to(device)
    y = model(x)
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ Batch=8 OK! VRAM dùng: {used:.1f} GB / {total:.0f} GB')
    print(f'   Còn trống: {total-used:.1f} GB (nên ≥ 2 GB)')
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print('⚠️  VRAM không đủ cho batch=8!')
        print('   → Sửa batch_size_per_gpu thành 4 trong Ô 6 rồi chạy lại')
    else:
        raise
finally:
    del model, x
    torch.cuda.empty_cache()

## Ô 8: Setup thư mục checkpoint + Save về Drive
Ô này tạo script auto-save mỗi khi Colab sắp hết session

In [ ]:
import os

# Thư mục experiment trên Colab
EXP_DIR    = '/content/NAFNet/experiments/NAFNet-Cards-width32'
MODELS_DIR = f'{EXP_DIR}/models'
STATES_DIR = f'{EXP_DIR}/training_states'

# Thư mục lưu trên Drive
DRIVE_DIR  = '/content/drive/MyDrive/deblur_cards_project/checkpoints_fold0'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(STATES_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR,  exist_ok=True)

print(f'✅ Thư mục tạo xong:')
print(f'   Colab: {EXP_DIR}')
print(f'   Drive: {DRIVE_DIR}')

# Hàm backup nhanh lên Drive (gọi thủ công khi cần)
def backup_to_drive():
    import shutil, glob, re
    pths   = glob.glob(f'{MODELS_DIR}/net_g_*.pth')
    states = glob.glob(f'{STATES_DIR}/*.state')
    count  = 0
    for f in pths + states:
        dst = os.path.join(DRIVE_DIR, os.path.basename(f))
        shutil.copy2(f, dst)
        count += 1
    # Tìm iter lớn nhất
    iters = [int(re.search(r'(\d+)\.pth', f).group(1)) for f in pths if re.search(r'(\d+)\.pth', f)]
    latest = max(iters) if iters else 0
    print(f'✅ Đã backup {count} files lên Drive. Latest iter: {latest}')
    return latest

print()
print('Để backup thủ công: gọi backup_to_drive()')

## Ô 9: (Chỉ khi resume) Restore checkpoint từ Drive
Bỏ qua ô này nếu train từ đầu. Chỉ chạy khi resume sau khi Colab bị disconnect.

In [ ]:
import shutil, os, re, glob

DRIVE_DIR  = '/content/drive/MyDrive/deblur_cards_project/checkpoints_fold0'
MODELS_DIR = '/content/NAFNet/experiments/NAFNet-Cards-width32/models'
STATES_DIR = '/content/NAFNet/experiments/NAFNet-Cards-width32/training_states'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(STATES_DIR, exist_ok=True)

# Copy từ Drive về Colab
for f in os.listdir(DRIVE_DIR):
    if f.endswith('.state'):
        shutil.copy(os.path.join(DRIVE_DIR, f), os.path.join(STATES_DIR, f))
        print(f'  ✅ Restored state: {f}')
    elif f.endswith('.pth') and 'net_g' in f:
        shutil.copy(os.path.join(DRIVE_DIR, f), os.path.join(MODELS_DIR, f))
        print(f'  ✅ Restored model: {f}')

# Tìm state iter lớn nhất
states = glob.glob(f'{STATES_DIR}/*.state')
if not states:
    print('⚠️  Không tìm thấy state file → sẽ train từ đầu')
else:
    latest = max(states, key=lambda x: int(re.search(r'(\d+)\.state', x).group(1)))
    latest_iter = int(re.search(r'(\d+)\.state', latest).group(1))
    print(f'\n▶️  Sẽ resume từ iter: {latest_iter}')

    # Cập nhật config để resume
    with open('/content/NAFNet/options/train/nafnet_custom.yml', 'r') as f:
        cfg = f.read()
    cfg = re.sub(r'resume_state:.*', f'resume_state: {latest}', cfg)
    with open('/content/NAFNet/options/train/nafnet_custom.yml', 'w') as f:
        f.write(cfg)
    print(f'✅ Config cập nhật resume_state → {latest}')
    print('Chạy Ô 10 để tiếp tục train!')

## Ô 10: 🚀 TRAIN!
- Lần đầu: ~2-3 giờ trên T4 (100k iter × ~0.09s/iter)
- Colab Free sẽ disconnect sau ~3-4h. Dùng Ô 9 để resume.
- Checkpoint tự lưu mỗi 5000 iter vào `/experiments/.../models/`
- **SAU KHI TRAIN XONG** hoặc **TRƯỚC KHI COLAB DISCONNECT**: chạy `backup_to_drive()`

In [ ]:
%cd /content/NAFNet

!torchrun \
    --nproc_per_node=1 \
    --master_port=4321 \
    basicsr/train.py \
    -opt options/train/nafnet_custom.yml \
    --launcher pytorch 2>&1 | tee /content/train_log.txt

## Ô 11: Backup checkpoint lên Drive sau khi train

In [ ]:
# Chạy ô này ngay khi training xong hoặc khi Colab sắp hết session
latest_iter = backup_to_drive()
print(f'✅ Checkpoint đến iter {latest_iter} đã được lưu an toàn trên Drive')

## Ô 12: Export best model ra file .pth sẵn để dùng

In [ ]:
import torch, glob, re, shutil, os

MODELS_DIR = '/content/NAFNet/experiments/NAFNet-Cards-width32/models'
DRIVE_DIR  = '/content/drive/MyDrive/deblur_cards_project/checkpoints_fold0'

# Tìm tất cả checkpoint
pths = glob.glob(f'{MODELS_DIR}/net_g_*.pth')
if not pths:
    # Thử tìm trên Drive
    pths = glob.glob(f'{DRIVE_DIR}/net_g_*.pth')

if not pths:
    print('❌ Không tìm thấy checkpoint nào!')
else:
    # Lấy checkpoint iter lớn nhất
    latest_pth = max(pths, key=lambda x: int(re.search(r'(\d+)\.pth', x).group(1)))
    latest_iter = int(re.search(r'(\d+)\.pth', latest_pth).group(1))
    print(f'Latest checkpoint: iter {latest_iter}')

    # Load và re-save với key 'params' (tương thích với step5/step6)
    ckpt = torch.load(latest_pth, map_location='cpu')
    # BasicSR lưu dưới 'params_ema' hoặc 'params'
    if 'params_ema' in ckpt:
        state_dict = ckpt['params_ema']
        print('Dùng params_ema (better quality)')
    elif 'params' in ckpt:
        state_dict = ckpt['params']
        print('Dùng params')
    else:
        state_dict = ckpt
        print('Dùng full state dict')

    # Lưu file clean
    clean_ckpt = {'params': state_dict}
    out_path   = f'{DRIVE_DIR}/nafnet_cards_best_v2.pth'
    torch.save(clean_ckpt, out_path)

    size = os.path.getsize(out_path) / 1e6
    print(f'\n✅ Đã lưu: {out_path} ({size:.1f} MB)')
    print(f'   Dùng file này trong step5_demo.py và step6_app.py')
    print(f'   python scripts/step5_demo.py --checkpoint pretrained/nafnet_cards_best_v2.pth ...')

## Ô 13: Đánh giá nhanh PSNR/SSIM trên tập test
Chạy sau khi training xong để có con số cho báo cáo

In [ ]:
import torch, cv2, numpy as np, glob, sys
from pathlib import Path

sys.path.insert(0, '/content/NAFNet')
from basicsr.models.archs.NAFNet_arch import NAFNet

DRIVE_DIR   = '/content/drive/MyDrive/deblur_cards_project/checkpoints_fold0'
CKPT_PATH   = f'{DRIVE_DIR}/nafnet_cards_best_v2.pth'
TEST_BLUR   = '/content/deblur_cards/data/test/blur'
TEST_SHARP  = '/content/deblur_cards/data/test/sharp'
MAX_EVAL    = 200   # Đánh giá tối đa bao nhiêu ảnh

# Load model
device = torch.device('cuda')
model  = NAFNet(img_channel=3, width=32, middle_blk_num=12,
                enc_blk_nums=[2,2,4,8], dec_blk_nums=[2,2,2,2])
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt.get('params', ckpt), strict=False)
model.to(device).eval()
print(f'✅ Model loaded')

# Metrics
def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64))**2)
    return 100.0 if mse == 0 else 20 * np.log10(255. / np.sqrt(mse))

def ssim_simple(a, b):
    try:
        from skimage.metrics import structural_similarity
        return structural_similarity(a, b, data_range=255, channel_axis=2)
    except:
        return 0.0

blur_files = sorted(glob.glob(f'{TEST_BLUR}/*.png'))[:MAX_EVAL]
pb_list, sb_list, pa_list, sa_list = [], [], [], []

print(f'Đánh giá {len(blur_files)} ảnh...')
for i, bf in enumerate(blur_files):
    sf = bf.replace('/blur/', '/sharp/')
    if not Path(sf).exists():
        continue
    sharp = cv2.imread(sf)
    blur  = cv2.imread(bf)
    if sharp is None or blur is None:
        continue
    # Inference
    t = torch.from_numpy(blur/255.).permute(2,0,1).float().unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(t)
    result = (out.squeeze().permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)

    pb_list.append(psnr(sharp, blur))
    sb_list.append(ssim_simple(sharp, blur))
    pa_list.append(psnr(sharp, result))
    sa_list.append(ssim_simple(sharp, result))

    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(blur_files)}...')

avg = lambda x: sum(x)/len(x) if x else 0
print()
print('═'*52)
print('  KẾT QUẢ ĐÁNH GIÁ')
print('═'*52)
print(f'  Số ảnh: {len(pa_list)}')
print(f'  {"Metric":<12} {"Trước":>12} {"Sau":>12} {"Cải thiện":>12}')
print(f'  {"-"*50}')
print(f'  {"PSNR (dB)":<12} {avg(pb_list):>12.2f} {avg(pa_list):>12.2f} {avg(pa_list)-avg(pb_list):>+11.2f}')
print(f'  {"SSIM":<12} {avg(sb_list):>12.4f} {avg(sa_list):>12.4f} {avg(sa_list)-avg(sb_list):>+11.4f}')
print('═'*52)

## Ô 14: Visualize kết quả (xem ảnh so sánh)
Chạy ô này để xem bằng mắt trước khi download model

In [ ]:
import torch, cv2, numpy as np, sys, random
from pathlib import Path
import matplotlib.pyplot as plt

sys.path.insert(0, '/content/NAFNet')
from basicsr.models.archs.NAFNet_arch import NAFNet

DRIVE_DIR  = '/content/drive/MyDrive/deblur_cards_project/checkpoints_fold0'
CKPT_PATH  = f'{DRIVE_DIR}/nafnet_cards_best_v2.pth'
TEST_BLUR  = '/content/deblur_cards/data/test/blur'
TEST_SHARP = '/content/deblur_cards/data/test/sharp'
N_SHOW     = 4   # Số ảnh hiển thị

# Load model
device = torch.device('cuda')
model  = NAFNet(img_channel=3, width=32, middle_blk_num=12,
                enc_blk_nums=[2,2,4,8], dec_blk_nums=[2,2,2,2])
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt.get('params', ckpt), strict=False)
model.to(device).eval()

import glob
all_blurs = sorted(glob.glob(f'{TEST_BLUR}/*.png'))
samples   = random.sample(all_blurs, min(N_SHOW, len(all_blurs)))

fig, axes = plt.subplots(N_SHOW, 3, figsize=(15, 5*N_SHOW))
if N_SHOW == 1: axes = [axes]

for row, bf in enumerate(samples):
    sf = bf.replace('/blur/', '/sharp/')
    blur  = cv2.imread(bf)
    sharp = cv2.imread(sf) if Path(sf).exists() else None

    t = torch.from_numpy(blur/255.).permute(2,0,1).float().unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(t)
    result = (out.squeeze().permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)

    axes[row][0].imshow(cv2.cvtColor(blur,   cv2.COLOR_BGR2RGB)); axes[row][0].set_title('BLUR (input)', fontsize=12)
    axes[row][1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)); axes[row][1].set_title('DEBLURRED (model output)', fontsize=12)
    if sharp is not None:
        axes[row][2].imshow(cv2.cvtColor(sharp, cv2.COLOR_BGR2RGB)); axes[row][2].set_title('SHARP (ground truth)', fontsize=12)
    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
plt.savefig('/content/deblur_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print('✅ Đã lưu: /content/deblur_visualization.png')